# SXMU_MDD dTMS 106 例 E2E 验证 — sros-sdk DSL

> **DSL-6** | Phase 9 AI4S DSL 科学编译器  
> 用 sros-sdk DSL 重写 dTMS 106 例 GNN 训练管线  
> 验证 `BrainGraphDataset.from_duckdb()` → `GNNTrainer.train()` 端到端链路

## 运行环境

- **Demo 模式**（默认）：使用合成数据，无需 graphmri CLI，纯 Python 可运行
- **真实模式**：需要 DuckDB 数据库 + graphmri CLI + HPC 集群

```bash
# Demo 模式
export SROS_SDK_DEMO=1
cd sros-sdk && PYTHONPATH=src jupyter notebook examples/sxmu_mdd_dtms_e2e.ipynb

# 真实模式（需要 SXMU 集群访问）
unset SROS_SDK_DEMO
cd sros-sdk && PYTHONPATH=src jupyter notebook examples/sxmu_mdd_dtms_e2e.ipynb
```

## 0. 环境准备

In [ ]:
import os
import sys
import numpy as np
import duckdb

# Demo mode: synthetic data for offline testing
os.environ["SROS_SDK_DEMO"] = "1"

from sros.brain import BrainGraphDataset
from sros.ml import GNNTrainer, InfraStrategy

print(f"Demo mode: {os.environ.get('SROS_SDK_DEMO', '0')}")
print(f"InfraStrategy: {GNNTrainer._detect_infra().value}")

## 1. 创建模拟 DuckDB 数据库

模拟 SXMU_MDD DuckDB 8 表 schema（subjects + phenotypes 子集），
插入 106 例 dTMS 队列的合成数据。

In [ ]:
import os as _os
# Clean up leftover from previous run
DB_PATH = "/tmp/sxmu_mdd_demo.db"
if _os.path.exists(DB_PATH):
    _os.remove(DB_PATH)

import duckdb

conn = duckdb.connect(DB_PATH)

conn.execute("""
    CREATE TABLE subjects (
        subject_id VARCHAR PRIMARY KEY,
        bids_path VARCHAR,
        intervention_type VARCHAR,
        age INTEGER,
        sex VARCHAR
    )
""")

conn.execute("""
    CREATE TABLE phenotypes (
        subject_id VARCHAR,
        phenotype_HAMD_improvement_pct DOUBLE,
        phenotype_HAMD_baseline DOUBLE,
        phenotype_response INTEGER,
        FOREIGN KEY (subject_id) REFERENCES subjects(subject_id)
    )
""")

# 插入 106 例 dTMS 被试
rng = np.random.default_rng(42)
for i in range(106):
    sid = f"sub-{i + 1:03d}"
    bids = f"/data/sxmu_mdd/bids/{sid}"
    age = int(rng.integers(10, 24))
    sex = rng.choice(["M", "F"])
    hamd_imp = round(float(rng.uniform(5, 85)), 1)
    hamd_base = round(float(rng.uniform(14, 35)), 1)
    responder = 1 if hamd_imp >= 50 else 0
    
    conn.execute(
        "INSERT INTO subjects VALUES (?, ?, 'dTMS', ?, ?)",
        [sid, bids, age, sex]
    )
    conn.execute(
        "INSERT INTO phenotypes VALUES (?, ?, ?, ?)",
        [sid, hamd_imp, hamd_base, responder]
    )

conn.close()

# 验证
conn = duckdb.connect(DB_PATH, read_only=True)
count = conn.execute("SELECT COUNT(*) FROM subjects WHERE intervention_type = 'dTMS'").fetchone()[0]
print(f"dTMS cohort size: {count} subjects (expected: 106)")

# 展示前 5 例
rows = conn.execute("""
    SELECT s.subject_id, s.age, s.sex, p.phenotype_HAMD_improvement_pct, p.phenotype_response
    FROM subjects s JOIN phenotypes p ON s.subject_id = p.subject_id
    WHERE s.intervention_type = 'dTMS'
    LIMIT 5
""").fetchall()
for r in rows:
    print(f"  {r[0]} | age={r[1]} | sex={r[2]} | HAMD_imp={r[3]:.1f}% | responder={r[4]}")

conn.close()

## 2. Step 1: from_duckdb() — 加载 dTMS 106 例队列

DSL 链式调用第一步：从 DuckDB 查询被试列表。
SQL JOIN 自动关联 subjects 和 phenotypes 表。

In [ ]:
cohort = BrainGraphDataset.from_duckdb(
    "SELECT s.*, p.phenotype_HAMD_improvement_pct, "
    "p.phenotype_HAMD_baseline, p.phenotype_response "
    "FROM subjects s JOIN phenotypes p ON s.subject_id = p.subject_id "
    "WHERE s.intervention_type = 'dTMS'",
    db_path=DB_PATH,
)

print(f"Cohort size: {len(cohort.subjects)} subjects")
print(f"First subject: {cohort.subjects[0]['subject_id']}")
print(f"Phenotype keys: {list(cohort.subjects[0].get('phenotypes', {}).keys())}")

## 3. Step 2: apply_pipeline() — 应用 fMRIPrep 预处理管线

In [ ]:
cohort = cohort.apply_pipeline(
    "fmriprep",
    resolution="2mm",
    output_dir="/data/sxmu_mdd/derivatives/dsl_test"
)

print(f"Pipeline: {cohort._pipeline}")
print(f"Pipeline kwargs: {cohort._pipeline_kwargs}")
print("\u2705 apply_pipeline: parameters stored, ready for deferred execution")

## 4. Step 3: extract_connectome() — 提取 AAL 脑连接矩阵

In [ ]:
cohort = cohort.extract_connectome(
    atlas="aal",          # AAL 116 区脑图谱
    kind="correlation",   # 皮尔逊相关矩阵
)

print(f"Atlas: {cohort._atlas}")
print(f"Connectome kind: {cohort._kind}")
print(f"Expected regions: {cohort._atlas_region_count()}")
print(f"Expected edges (lower triangle): {cohort._atlas_region_count() * (cohort._atlas_region_count() - 1) // 2}")
print("\u2705 extract_connectome: parameters stored, waiting for .compute()")

## 5. Step 4: compute() — 执行连接组计算

**Demo 模式**：生成合成 SPD（对称正定）连接矩阵  
**真实模式**：调用 `graphmri --raw build-network` CLI

In [ ]:
cohort = cohort.compute(demo=True)  # Demo: synthetic data

print(f"Connectome data shape: {cohort._connectome_data.shape}")
print(f"  Subjects: {cohort._connectome_data.shape[0]}")
print(f"  Edges per subject: {cohort._connectome_data.shape[1]}")
print(f"Labels shape: {cohort._labels.shape if cohort._labels is not None else 'None'}")
print(f"Label mean: {cohort._labels.mean():.1f}% HAMD improvement")

# DSPy 验证: 检查全矩阵的对称正定性
if hasattr(cohort, '_connectome_matrices') and cohort._connectome_matrices is not None:
    mat = cohort._connectome_matrices[0]
    is_sym = np.allclose(mat, mat.T, atol=1e-10)
    eigvals = np.linalg.eigvalsh(mat)
    is_pd = np.all(eigvals > 0)
    print(f"\nDSPy validation (subject 0):")
    print(f"  Symmetric: {is_sym}")
    print(f"  Positive definite: {is_pd} (min eigval: {eigvals.min():.4f})")
    print("\u2705 DSPy assertions: PASSED")

## 6. Step 5: to_dataloader() — 转换为 PyTorch DataLoader

In [ ]:
import torch

loader = cohort.to_dataloader(batch_size=16, shuffle=True)

print(f"Batch size: {loader.batch_size}")
print(f"Number of batches: {sum(1 for _ in loader)}")

# 取第一批验证 shape
for features, labels in loader:
    print(f"Features shape: {features.shape}")
    print(f"Labels shape: {labels.shape}")
    break

print("\u2705 to_dataloader: ready for model training")

## 7. Step 6: GNNTrainer.train() — 训练 GNN 预测 HAMD 改善率

In [ ]:
trainer = GNNTrainer(cohort)

result = trainer.train(
    model="random_forest",
    target="HAMD_improvement_pct",  # 治疗后 HAMD 改善百分比
    infra_strategy="auto",          # 自动检测 GPU/Slurm
)

print(f"\n=== Training Results ===")
print(f"Model: {result.get('model', 'random_forest')}")
print(f"R\u00b2 Score: {result['accuracy']:.4f}")
print(f"ROC AUC: {result['roc_auc']:.4f}")
print(f"Device: {result['device']}")
print(f"Strategy: {result['strategy']}")
print(f"Strategy reason: {result['strategy_reason']}")
print(f"CV scores (5-fold): {[f'{s:.3f}' for s in result['cv_scores']]}")

## 8. DSL 完整链式调用（一句话版本）

等价于以上 6 个步骤的 Fluent API 链式调用。

In [ ]:
result = (
    GNNTrainer(
        BrainGraphDataset
        .from_duckdb(
            "SELECT s.*, p.phenotype_HAMD_improvement_pct, "
            "p.phenotype_HAMD_baseline, p.phenotype_response "
            "FROM subjects s JOIN phenotypes p ON s.subject_id = p.subject_id "
            "WHERE s.intervention_type = 'dTMS'",
            db_path=DB_PATH,
        )
        .apply_pipeline("fmriprep", resolution="2mm")
        .extract_connectome(atlas="aal", kind="correlation")
        .compute(demo=True)
    )
    .train(
        model="random_forest",
        target="HAMD_improvement_pct",
        infra_strategy="auto",
    )
)

print(f"R\u00b2 Score: {result['accuracy']:.4f}")
print(f"MAE: {result.get('mae', 'N/A')}")
print(f"Strategy: {result['strategy']} ({result['strategy_reason']})")
print("\n\u2705 DSL-6: SXMU_MDD dTMS 106 E2E pipeline verified")

## 9. DSL vs 原生 CLI 等价性验证

验证 DSL 输出与 `graphmri --raw predict` CLI 输出的 JSON schema 一致性。

In [ ]:
# DSL 输出键契约
dsl_keys = set(result.keys())
required_keys = {"accuracy", "roc_auc", "device", "strategy", "strategy_reason"}

print(f"Required keys present: {required_keys.issubset(dsl_keys)}")
print(f"DSL keys: {sorted(dsl_keys)}")

# graphmri CLI --raw predict 等价 JSON 结构:
cli_equivalent = {
    "accuracy": result["accuracy"],
    "roc_auc": result["roc_auc"],
    "device": result["device"],
}

assert isinstance(result["accuracy"], float)
assert isinstance(result["roc_auc"], float)
print("\n\u2705 DSL-graphmri CLI contract: compatible")

## 10. 清理

In [ ]:
# 清理临时数据库
os.remove(DB_PATH)
print(f"Cleaned up {DB_PATH}")